# 01 · Baseline and calibration (ResNet-50, Colab T4)

Trains the baseline on the leak-free `source` split, then on the `supplied` split for the
leakage comparison. All code lives in `src/` of this repo; this notebook only calls it, so
Colab runs exactly the code the tests ran.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Put
`grape_dataset_for_colab.zip` at the top level of My Drive. Then Runtime → Run all.

Every output goes to Google Drive (`OUT`) as it is produced, with the model, split and a
timestamp in its filename. See `HANDOVER.md` for timings and the files to check.

In [ ]:
# ---- settings: the only cell you normally edit ---------------------------------
REPO_URL = "https://github.com/brycedaniliuc/COMP2035-grape-calibration.git"
REPO_REF = "main"                       # a branch or tag, e.g. "v1.0"
ZIP_PATH = "/content/drive/MyDrive/grape_dataset_for_colab.zip"   # must match Drive
DATA_ROOT = "/content/grape_dataset"    # the zip extracts to this folder
OUT = "/content/drive/MyDrive/COMP2035_runs"   # every output is written here, as it goes

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
T0 = time.time()

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

REPO = Path("/content/COMP2035-grape-calibration")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "--tags"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True,
                        capture_output=True, text=True).stdout.strip()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("code at", REPO_REF, commit)

# Colab already has torch, torchvision, numpy, scikit-learn, scikit-image and matplotlib.
# Do NOT pip install requirements.txt here: it pins the CPU build of torch.
import importlib.util
missing = [m for m in ("skimage", "sklearn", "nbformat") if importlib.util.find_spec(m) is None]
if missing:
    pkg = {"skimage": "scikit-image", "sklearn": "scikit-learn"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[pkg.get(m, m) for m in missing]], check=True)

from src.train import environment, select_device
import json
env = environment(select_device())
print(json.dumps(env, indent=1))
if env["device"] != "cuda":
    print("!! NO GPU: Runtime > Change runtime type > T4 GPU, then run all again")
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)

In [ ]:
# Fail early: unzip and check the exact per-folder counts BEFORE any training.
import shutil, zipfile
if not os.path.isdir(DATA_ROOT):
    assert os.path.exists(ZIP_PATH), (
        f"{ZIP_PATH} not found. Put grape_dataset_for_colab.zip at the top level of My Drive. "
        f"Files there: {sorted(os.listdir('/content/drive/MyDrive'))[:40]}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/")
from src.data import check_supplied_dataset
check_supplied_dataset(DATA_ROOT)       # raises unless 7,222 train + 1,805 test, per class
print(f"elapsed {(time.time() - T0) / 60:.1f} min")

## Dataset audit and leakage of the supplied split

In [ ]:
from datetime import datetime
from src.audit import leaking_leaves, print_audit, supplied_split_audit
from src.data import index_dataset
records = index_dataset(DATA_ROOT)
audit = supplied_split_audit(records)
print_audit(audit, leaking_leaves(records))
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
with open(f"{OUT}/audit_{stamp}.json", "w") as fh:
    json.dump(audit, fh, indent=2)
print("written ->", f"{OUT}/audit_{stamp}.json")

## Train on the `source` split (about 26 min on a T4)

The checkpoint is saved to Drive whenever validation loss improves, so an interrupted run
still leaves the best weights behind.

In [ ]:
from src.config import Config
from src.train import run
cfg_source = Config(data_root=DATA_ROOT, out_dir=OUT, split_mode="source", model="resnet50")
res_source = run(cfg_source)
SOURCE_RESULTS = f"{OUT}/results_{res_source['run_id']}.json"
print("source run:", SOURCE_RESULTS, f"| elapsed {(time.time() - T0) / 60:.1f} min")

## Verify the source run (Tier 1 must pass; Tier 2 compares with September)

In [ ]:
r = subprocess.run([sys.executable, "-m", "src.verify", "--run", SOURCE_RESULTS],
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
assert r.returncode == 0, "a Tier 1 check failed: stop and investigate before using this run"

## Train on the `supplied` split: the leakage comparison (about 18 min)

In [ ]:
cfg_supplied = Config(data_root=DATA_ROOT, out_dir=OUT, split_mode="supplied",
                      model="resnet50")
res_supplied = run(cfg_supplied)
print("supplied run:", f"{OUT}/results_{res_supplied['run_id']}.json",
      f"| elapsed {(time.time() - T0) / 60:.1f} min")

## Everything this notebook wrote to Drive

In [ ]:
for p in sorted(Path(OUT).iterdir()):
    print(f"{p.stat().st_size / 1e6:9.2f} MB  {p.name}")